# Production-Grade Error Handling and Retry Patterns for the OpenAI API

This notebook teaches you how to build resilient applications on top of the OpenAI API.  
Real production systems encounter rate limits, network blips, and transient server errors. Handling them gracefully — rather than letting them surface as crashes — is what separates toy demos from reliable products.

**What you will learn:**
- How to classify OpenAI SDK errors and decide when to retry vs. fail fast
- Exponential back-off with the `tenacity` library (sync and async)
- How to read rate-limit headers to stay ahead of 429s
- Timeout configuration at the client and per-request level
- A fallback-model strategy for capacity spikes
- Structured error logging that captures enough context to debug later

> **All runnable cells use mocked responses** so you can run this notebook without an API key. Where real API calls would occur, the mock is clearly labelled.

## 1. Setup

Install the two libraries used throughout this notebook.

In [ ]:
%pip install openai tenacity --quiet

In [ ]:
import os
import time
import asyncio
import logging
import json
from typing import Optional, List, Dict, Any
from unittest.mock import MagicMock, patch

import httpx

from openai import OpenAI, AsyncOpenAI
from openai import (
    RateLimitError,
    APIError,
    APITimeoutError,
    APIConnectionError,
    AuthenticationError,
)
from tenacity import (
    retry,
    wait_exponential,
    stop_after_attempt,
    retry_if_exception_type,
    before_sleep_log,
    RetryError,
)

logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")
logger = logging.getLogger(__name__)

print("openai and tenacity imported successfully.")

openai and tenacity imported successfully.


## 2. Error Taxonomy

The OpenAI Python SDK maps HTTP status codes and network-level failures to specific exception classes. Understanding what each one means lets you choose the right recovery strategy immediately rather than applying a blanket retry to everything.

| Exception class | HTTP status | Root cause | Retry? | Strategy |
|---|---|---|---|---|
| `RateLimitError` | 429 | Too many requests (RPM/TPM/RPD limit hit) | **Yes** | Exponential back-off; inspect `x-ratelimit-reset-requests` header |
| `APIError` | 5xx | Transient server-side failure | **Yes** | Exponential back-off with a cap (e.g. 6 attempts) |
| `APITimeoutError` | — | Request exceeded configured timeout | **Yes (carefully)** | Retry with increased timeout or shorter prompt |
| `APIConnectionError` | — | DNS / TLS / socket failure before response | **Yes** | Short fixed delay then retry; check network |
| `AuthenticationError` | 401 | Bad or missing API key | **No** | Fail fast — retrying wastes quota |
| `BadRequestError` | 400 | Invalid parameters (e.g. prompt too long) | **No** | Fix the request; retrying will always fail |
| `NotFoundError` | 404 | Model or resource does not exist | **No** | Fail fast — the model ID is wrong |
| `PermissionDeniedError` | 403 | Key lacks access to this model/org | **No** | Fail fast — contact OpenAI support |

**Rule of thumb:** Retry on `RateLimitError`, `APIError` (5xx), `APITimeoutError`, and `APIConnectionError`. Fail fast on everything else.

## 3. Basic Retry with Tenacity

`tenacity` is a retry library that keeps retry logic out of your business code and in a declarative decorator.  
The settings below implement standard best practices:
- **Exponential back-off** (`wait_exponential`): waits 1 s, 2 s, 4 s, 8 s … up to 60 s between attempts.
- **Attempt cap** (`stop_after_attempt(6)`): prevents infinite loops if the API is truly down.
- **Selective retry** (`retry_if_exception_type`): only retries transient errors — does not swallow programming mistakes.

In [ ]:
# Real client reads OPENAI_API_KEY from environment — no hardcoded secrets.
# client = OpenAI()  # uncomment when running against the real API

RETRYABLE_ERRORS = (RateLimitError, APIError, APITimeoutError, APIConnectionError)


@retry(
    wait=wait_exponential(multiplier=1, min=1, max=60),
    stop=stop_after_attempt(6),
    retry=retry_if_exception_type(RETRYABLE_ERRORS),
    before_sleep=before_sleep_log(logger, logging.WARNING),
    reraise=True,
)
def chat_with_retry(
    client: OpenAI,
    messages: List[Dict[str, str]],
    model: str = "gpt-4o-mini",
    **kwargs: Any,
) -> str:
    """Call chat completions with automatic exponential back-off retry."""
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        **kwargs,
    )
    return response.choices[0].message.content


# --- Mock demonstration ---
mock_client = MagicMock(spec=OpenAI)
mock_response = MagicMock()
mock_response.choices[0].message.content = "Hello! How can I help you today?"
mock_client.chat.completions.create.return_value = mock_response

result = chat_with_retry(
    mock_client,
    messages=[{"role": "user", "content": "Hello"}],
)
print(f"Response (mocked): {result}")

# Simulate two RateLimitErrors then success
attempt_counter = {"n": 0}
mock_success = MagicMock()
mock_success.choices[0].message.content = "Hello! I am a mocked response after retries."


def _side_effect(*args, **kwargs):
    attempt_counter["n"] += 1
    if attempt_counter["n"] <= 2:
        print(f"Simulating RateLimitError on attempt {attempt_counter['n']} ...")
        # Build a minimal mock that looks like RateLimitError
        mock_req = MagicMock()
        mock_req.method = "POST"
        mock_req.url = "https://api.openai.com/v1/chat/completions"
        err = RateLimitError(
            message="Rate limit exceeded",
            response=MagicMock(status_code=429, headers={}),
            body={"error": {"message": "Rate limit exceeded"}},
        )
        raise err
    return mock_success


mock_client.chat.completions.create.side_effect = _side_effect

try:
    result2 = chat_with_retry(
        mock_client,
        messages=[{"role": "user", "content": "Hello again"}],
    )
    print(f"Response after retries (mocked): {result2}")
except Exception as exc:
    print(f"Failed after all retries: {exc}")

Response (mocked): Hello! How can I help you today?
Simulating RateLimitError on attempt 1 ...
Simulating RateLimitError on attempt 2 ...
Response after retries (mocked): Hello! I am a mocked response after retries.


## 4. Rate Limit Header Inspection

The OpenAI API returns rate-limit metadata in response headers. Reading these headers lets you react *before* you hit a 429 rather than after.

Use `client.chat.completions.with_raw_response.create()` to get the raw `httpx.Response` and access its headers:

| Header | Meaning |
|---|---|
| `x-ratelimit-limit-requests` | Your RPM limit for this tier |
| `x-ratelimit-remaining-requests` | Requests available in the current window |
| `x-ratelimit-reset-requests` | ISO-8601 timestamp when the window resets |
| `x-ratelimit-limit-tokens` | Your TPM limit |
| `x-ratelimit-remaining-tokens` | Tokens available in the current window |
| `x-ratelimit-reset-tokens` | ISO-8601 timestamp when the token window resets |

In [ ]:
def call_with_header_inspection(
    client: OpenAI,
    messages: List[Dict[str, str]],
    model: str = "gpt-4o-mini",
) -> Dict[str, Any]:
    """
    Make a chat completion call and return both the content and
    the parsed rate-limit headers from the raw HTTP response.
    """
    raw = client.chat.completions.with_raw_response.create(
        model=model,
        messages=messages,
    )
    headers = raw.headers
    rate_info = {
        "limit-requests":     headers.get("x-ratelimit-limit-requests"),
        "remaining-requests": headers.get("x-ratelimit-remaining-requests"),
        "reset-requests":     headers.get("x-ratelimit-reset-requests"),
        "limit-tokens":       headers.get("x-ratelimit-limit-tokens"),
        "remaining-tokens":   headers.get("x-ratelimit-remaining-tokens"),
        "reset-tokens":       headers.get("x-ratelimit-reset-tokens"),
    }
    completion = raw.parse()
    content = completion.choices[0].message.content
    return {"content": content, "rate_info": rate_info}


# --- Mock demonstration ---
mock_headers = {
    "x-ratelimit-limit-requests":     "500",
    "x-ratelimit-remaining-requests": "347",
    "x-ratelimit-reset-requests":     "2026-06-21T12:00:00Z",
    "x-ratelimit-limit-tokens":       "200000",
    "x-ratelimit-remaining-tokens":   "198500",
    "x-ratelimit-reset-tokens":       "2026-06-21T12:00:01Z",
}
mock_raw = MagicMock()
mock_raw.headers = mock_headers
mock_completion = MagicMock()
mock_completion.choices[0].message.content = "This is a mocked chat completion response."
mock_raw.parse.return_value = mock_completion

mock_client2 = MagicMock(spec=OpenAI)
mock_client2.chat.completions.with_raw_response.create.return_value = mock_raw

result = call_with_header_inspection(
    mock_client2,
    messages=[{"role": "user", "content": "What is the capital of France?"}],
)

print("Rate limit headers from response:")
for k, v in result["rate_info"].items():
    print(f"  {k:<20}: {v}")
print(f"Content: {result['content']}")

Rate limit headers from response:
  limit-requests    : 500
  remaining-requests: 347
  reset-requests    : 2026-06-21T12:00:00Z
  limit-tokens      : 200000
  remaining-tokens  : 198500
  reset-tokens      : 2026-06-21T12:00:01Z
Content: This is a mocked chat completion response.


## 5. Proactive Rate Limit Tracking

Instead of waiting for a 429, you can track remaining quota after every call and proactively sleep when you are close to the limit. The `RateLimitTracker` class below maintains a running count of remaining requests and automatically inserts a delay when fewer than a configurable threshold remain.

This pattern is especially useful for batch processing workloads where you send hundreds of requests in a loop.

In [ ]:
class RateLimitTracker:
    """
    Wraps an OpenAI client to:
    1. Read rate-limit headers after every successful call.
    2. Sleep proactively when remaining requests fall below a threshold.
    """

    def __init__(
        self,
        client: OpenAI,
        low_water_mark: int = 20,
        sleep_seconds: float = 2.0,
    ):
        self.client = client
        self.low_water_mark = low_water_mark
        self.sleep_seconds = sleep_seconds
        self._remaining: Optional[int] = None

    def chat(
        self,
        messages: List[Dict[str, str]],
        model: str = "gpt-4o-mini",
        **kwargs: Any,
    ) -> str:
        raw = self.client.chat.completions.with_raw_response.create(
            model=model, messages=messages, **kwargs
        )
        remaining_str = raw.headers.get("x-ratelimit-remaining-requests")
        if remaining_str is not None:
            self._remaining = int(remaining_str)
        if self._remaining is not None and self._remaining < self.low_water_mark:
            logger.warning(
                "Rate limit close: %d requests remaining. Sleeping %.0fs before next call.",
                self._remaining,
                self.sleep_seconds,
            )
            time.sleep(self.sleep_seconds)
        completion = raw.parse()
        return completion.choices[0].message.content


# --- Mock demonstration: simulate decreasing remaining-requests ---
simulated_remaining = [50, 10, 5]
call_index = {"i": 0}


def _make_mock_raw(idx):
    r = MagicMock()
    r.headers = {"x-ratelimit-remaining-requests": str(simulated_remaining[idx])}
    c = MagicMock()
    c.choices[0].message.content = f"Response {idx + 1}"
    r.parse.return_value = c
    return r


def _tracker_side_effect(*args, **kwargs):
    idx = call_index["i"]
    call_index["i"] += 1
    return _make_mock_raw(idx)


mock_client3 = MagicMock(spec=OpenAI)
mock_client3.chat.completions.with_raw_response.create.side_effect = _tracker_side_effect

tracker = RateLimitTracker(mock_client3, low_water_mark=20, sleep_seconds=2.0)

# Override time.sleep to avoid actually waiting in the demo
original_sleep = time.sleep
time.sleep = lambda s: None  # no-op for demo

for i in range(3):
    content = tracker.chat(messages=[{"role": "user", "content": f"Question {i+1}"}])
    print(f"Call {i+1}: remaining={simulated_remaining[i]}, content={content}")

time.sleep = original_sleep  # restore
print("Batch complete.")

Call 1: remaining=50, content=Response 1
Call 2: remaining=10, content=Response 2
WARNING | Rate limit close: 10 requests remaining. Sleeping 2s before next call.
Call 3: remaining=5, content=Response 3
WARNING | Rate limit close: 5 requests remaining. Sleeping 2s before next call.
Batch complete.


## 6. Timeout Configuration

The default OpenAI client timeout is 10 minutes — long enough for edge cases to hang indefinitely in production. You should always set explicit timeouts.

`httpx.Timeout` takes three components:
- **connect**: time to establish the TCP connection
- **read**: time to receive bytes after the connection is open
- **write**: time to send the request body
- **pool**: time to acquire a connection from the connection pool

You can also override the timeout per-request using `client.with_options(timeout=...)`.

In [ ]:
# Pattern A: Set a sensible default on the client
# client = OpenAI(
#     timeout=httpx.Timeout(30.0, connect=5.0)
# )

# Pattern B: Override per request for long-running calls (e.g. large context)
# long_client = client.with_options(timeout=httpx.Timeout(60.0, connect=5.0))

# Pattern C: Handle APITimeoutError explicitly
def chat_with_timeout_handling(
    client: OpenAI,
    messages: List[Dict[str, str]],
    model: str = "gpt-4o-mini",
    timeout: float = 30.0,
) -> Optional[str]:
    try:
        response = client.chat.completions.create(
            model=model,
            messages=messages,
            timeout=timeout,
        )
        return response.choices[0].message.content
    except APITimeoutError as exc:
        logger.error("Request timed out after %.1fs: %s", timeout, exc)
        return None  # caller can decide to retry with a shorter prompt


# Demonstrate the timeout objects
default_timeout = httpx.Timeout(30.0, connect=5.0)
long_timeout    = httpx.Timeout(60.0, connect=5.0)
print(f"Client-level timeout: {default_timeout}")
print(f"Per-request timeout (long task): {long_timeout}")

# Simulate a timeout
mock_timeout_client = MagicMock(spec=OpenAI)
mock_timeout_client.chat.completions.create.side_effect = APITimeoutError(request=MagicMock())

result = chat_with_timeout_handling(
    mock_timeout_client,
    messages=[{"role": "user", "content": "Tell me everything about the universe."}],
    timeout=30.0,
)
assert result is None
print("APITimeoutError correctly raised when request times out.")

Client-level timeout: Timeout(connect=5.0, read=30.0, write=30.0, pool=30.0)
Per-request timeout (long task): Timeout(connect=5.0, read=60.0, write=60.0, pool=60.0)
APITimeoutError correctly raised when request times out.


## 7. Async Retry with AsyncOpenAI

For high-throughput workloads, `AsyncOpenAI` allows you to issue many concurrent calls with `asyncio.gather`. The `tenacity` library supports async functions natively — just use `@retry(...)` on an `async def` and `await` the calls.

A key benefit of `asyncio.gather` with per-task retry: each task retries independently. One task hitting a rate limit does not block the others.

In [ ]:
async_client = AsyncOpenAI  # Not instantiated — avoids needing a real API key


@retry(
    wait=wait_exponential(multiplier=1, min=1, max=60),
    stop=stop_after_attempt(6),
    retry=retry_if_exception_type(RETRYABLE_ERRORS),
    before_sleep=before_sleep_log(logger, logging.WARNING),
    reraise=True,
)
async def async_chat_with_retry(
    client: AsyncOpenAI,
    messages: List[Dict[str, str]],
    model: str = "gpt-4o-mini",
    **kwargs: Any,
) -> str:
    """Async chat completion with exponential back-off retry."""
    response = await client.chat.completions.create(
        model=model,
        messages=messages,
        **kwargs,
    )
    return response.choices[0].message.content


async def run_concurrent_batch(
    client: AsyncOpenAI,
    questions: List[str],
    model: str = "gpt-4o-mini",
) -> List[str]:
    """Issue all questions concurrently; each task retries independently."""
    tasks = [
        async_chat_with_retry(
            client,
            messages=[{"role": "user", "content": q}],
            model=model,
        )
        for q in questions
    ]
    return await asyncio.gather(*tasks, return_exceptions=False)


# --- Mock async demonstration ---
MOCK_ANSWERS = [
    "The sky appears blue due to Rayleigh scattering. (mocked)",
    "Water is a molecule of two hydrogen atoms and one oxygen atom. (mocked)",
    "Gravity is a force that attracts objects with mass toward each other. (mocked)",
]
answer_idx = {"i": 0}


async def _mock_create(*args, **kwargs):
    idx = answer_idx["i"] % len(MOCK_ANSWERS)
    answer_idx["i"] += 1
    mock_resp = MagicMock()
    mock_resp.choices[0].message.content = MOCK_ANSWERS[idx]
    return mock_resp


mock_async_client = MagicMock(spec=AsyncOpenAI)
mock_async_client.chat.completions.create = _mock_create

questions = [
    "Why is the sky blue?",
    "What is water made of?",
    "What is gravity?",
]

answers = asyncio.run(run_concurrent_batch(mock_async_client, questions))

print("Concurrent async results:")
for i, ans in enumerate(answers):
    print(f"  [{i}] {ans}")

Concurrent async results:
  [0] The sky appears blue due to Rayleigh scattering. (mocked)
  [1] Water is a molecule of two hydrogen atoms and one oxygen atom. (mocked)
  [2] Gravity is a force that attracts objects with mass toward each other. (mocked)


## 8. Fallback Model Strategy

During capacity events, your primary model (e.g. `gpt-4o`) may be rate-limited while cheaper or faster models remain available. A `FallbackClient` wrapper catches `RateLimitError` on the primary model and immediately retries on a fallback without waiting for back-off cycles.

This pattern is suitable when:
- Latency matters more than using the best model for every call.
- The fallback model (`gpt-4o-mini`) produces acceptable quality for the task.
- You want to keep throughput high during peak periods.

In [ ]:
class FallbackClient:
    """
    Wraps an OpenAI client with a primary → fallback model chain.
    On RateLimitError for the primary, retries immediately with the fallback.
    """

    def __init__(
        self,
        client: OpenAI,
        primary_model: str = "gpt-4o",
        fallback_model: str = "gpt-4o-mini",
    ):
        self.client = client
        self.primary_model = primary_model
        self.fallback_model = fallback_model

    def chat(
        self,
        messages: List[Dict[str, str]],
        **kwargs: Any,
    ) -> Dict[str, str]:
        """Returns {'model_used': ..., 'content': ...}."""
        try:
            response = self.client.chat.completions.create(
                model=self.primary_model,
                messages=messages,
                **kwargs,
            )
            return {
                "model_used": self.primary_model,
                "content": response.choices[0].message.content,
            }
        except RateLimitError:
            logger.warning(
                "%s rate limited. Falling back to %s.",
                self.primary_model,
                self.fallback_model,
            )
            response = self.client.chat.completions.create(
                model=self.fallback_model,
                messages=messages,
                **kwargs,
            )
            return {
                "model_used": self.fallback_model,
                "content": response.choices[0].message.content,
            }


# --- Mock demonstration ---
fallback_call_count = {"n": 0}
fallback_resp = MagicMock()
fallback_resp.choices[0].message.content = "Fallback response from gpt-4o-mini. (mocked)"
primary_resp = MagicMock()
primary_resp.choices[0].message.content = "Primary response from gpt-4o. (mocked)"


def _fallback_side_effect(*args, model="gpt-4o", **kwargs):
    fallback_call_count["n"] += 1
    if model == "gpt-4o" and fallback_call_count["n"] == 1:
        raise RateLimitError(
            message="You exceeded your current quota",
            response=MagicMock(status_code=429, headers={}),
            body={"error": {"message": "quota exceeded"}},
        )
    if model == "gpt-4o-mini":
        return fallback_resp
    return primary_resp


mock_fb_client = MagicMock(spec=OpenAI)
mock_fb_client.chat.completions.create.side_effect = _fallback_side_effect

fb = FallbackClient(mock_fb_client, primary_model="gpt-4o", fallback_model="gpt-4o-mini")

result1 = fb.chat(messages=[{"role": "user", "content": "Summarize this document."}])
print(f"model_used={result1['model_used']}, content={result1['content']}")

result2 = fb.chat(messages=[{"role": "user", "content": "Translate to French."}])
print(f"Primary available — model_used={result2['model_used']}, content={result2['content']}")

WARNING | gpt-4o rate limited. Falling back to gpt-4o-mini.
model_used=gpt-4o-mini, content=Fallback response from gpt-4o-mini. (mocked)
Primary available — model_used=gpt-4o, content=Primary response from gpt-4o. (mocked)


## 9. Structured Error Logging

When an error occurs in production you need enough context to reproduce and diagnose it. The OpenAI SDK provides:
- `error.status_code` — HTTP status
- `error.request_id` — unique ID from `x-request-id` header; include this when contacting OpenAI support
- `error.body` — JSON body with the `error.message` and `error.code` fields
- `error.response.headers` — all response headers, including rate-limit info

The `log_error` function below serialises all of this into a structured dict suitable for shipping to a log aggregator (e.g. Datadog, CloudWatch, Loki).

In [ ]:
def log_error(
    error: Exception,
    attempt: int,
    context: Optional[Dict[str, Any]] = None,
) -> Dict[str, Any]:
    """
    Build a structured log dict from an OpenAI SDK exception.
    Suitable for shipping to any log aggregator.

    Parameters
    ----------
    error   : The caught exception.
    attempt : Which retry attempt number this is (1-indexed).
    context : Optional caller-supplied dict (e.g. {'model': '...', 'user_id': '...'}).

    Returns
    -------
    A dict with all available diagnostic fields.
    """
    entry: Dict[str, Any] = {
        "error_type": type(error).__name__,
        "status_code": getattr(error, "status_code", None),
        "request_id": getattr(error, "request_id", None),
        "message": str(error),
        "error_code": None,
        "attempt": attempt,
        "context": context or {},
    }

    body = getattr(error, "body", None)
    if isinstance(body, dict) and "error" in body:
        entry["error_code"] = body["error"].get("code")
        if not entry["message"] or entry["message"] == str(error):
            entry["message"] = body["error"].get("message", entry["message"])

    response = getattr(error, "response", None)
    if response is not None:
        headers = getattr(response, "headers", {})
        entry["rate_limit_remaining"] = headers.get("x-ratelimit-remaining-requests")
        entry["rate_limit_reset"]     = headers.get("x-ratelimit-reset-requests")

    logger.error("OpenAI API error: %s", json.dumps(entry, default=str))
    return entry


# --- Mock demonstration ---
mock_error_response = MagicMock()
mock_error_response.status_code = 429
mock_error_response.headers = {
    "x-request-id":                  "req_abc123xyz",
    "x-ratelimit-remaining-requests": "0",
    "x-ratelimit-reset-requests":     "2026-06-21T12:05:00Z",
}

mock_rate_error = RateLimitError(
    message="You exceeded your current quota, please check your plan and billing details.",
    response=mock_error_response,
    body={
        "error": {
            "message": "You exceeded your current quota, please check your plan and billing details.",
            "type": "insufficient_quota",
            "code": "insufficient_quota",
        }
    },
)
# Attach request_id that the SDK normally parses from headers
mock_rate_error.request_id = "req_abc123xyz"

entry = log_error(
    mock_rate_error,
    attempt=3,
    context={"model": "gpt-4o-mini", "user_id": "user_42"},
)
print("Structured error log entry:")
print(json.dumps(entry, indent=2, default=str))

Structured error log entry:
{
  "error_type": "RateLimitError",
  "status_code": 429,
  "request_id": "req_abc123xyz",
  "message": "You exceeded your current quota, please check your plan and billing details.",
  "error_code": "insufficient_quota",
  "attempt": 3,
  "context": {
    "model": "gpt-4o-mini",
    "user_id": "user_42"
  },
  "rate_limit_remaining": "0",
  "rate_limit_reset": "2026-06-21T12:05:00Z"
}


## 10. Summary: Decision Tree

Use this decision tree to pick the right pattern for your situation.

```
Did the call fail?
├── No  →  Done. (Optionally read headers to track quota.)
└── Yes →  What error type?
    ├── AuthenticationError / BadRequestError / NotFoundError / PermissionDeniedError
    │   └── FAIL FAST — do not retry. Fix the code or config.
    ├── APITimeoutError
    │   └── Retry with tenacity (≤ 3 attempts). Consider reducing prompt size.
    ├── APIConnectionError
    │   └── Retry with tenacity. Check network connectivity.
    ├── APIError (5xx)
    │   └── Retry with exponential back-off (≤ 6 attempts, max 60s wait).
    └── RateLimitError (429)
        ├── One-off call  →  Retry with exponential back-off (tenacity).
        ├── Batch workload →  Use RateLimitTracker to sleep proactively.
        └── Latency-sensitive →  Use FallbackClient to switch to gpt-4o-mini.
```

### Pattern reference table

| Pattern | Module / Class | Best for |
|---|---|---|
| Basic retry | `chat_with_retry` (§3) | Any synchronous call |
| Header inspection | `call_with_header_inspection` (§4) | Observability and debugging |
| Proactive throttling | `RateLimitTracker` (§5) | High-volume batch jobs |
| Timeout control | `httpx.Timeout` + `with_options` (§6) | Preventing hanging requests |
| Async concurrent retry | `async_chat_with_retry` + `asyncio.gather` (§7) | High-throughput async workloads |
| Model fallback | `FallbackClient` (§8) | Latency-sensitive calls during capacity events |
| Structured logging | `log_error` (§9) | Production observability |

### Key takeaways

1. **Never retry authentication or parameter errors** — they will always fail.
2. **Exponential back-off + jitter** is the industry-standard approach for transient errors.
3. **Read the rate-limit headers** on every response if you send more than a handful of calls per minute.
4. **Set explicit timeouts** on the client and per-request rather than relying on the SDK default.
5. **Log `request_id`** — it is the single most useful field when filing a support ticket with OpenAI.